# System 1 Notebook 00 — Master ingestion & batch assignment

Notebook này là bước đầu tiên của System 1.

Nhiệm vụ chính:

1. Chuẩn bị runtime cho Kaggle / Colab / local.

2. Clone hoặc dùng repo `system1` hiện có.

3. Cấu hình input / output / artifact store.

4. Chạy phase00:

   * ingest dữ liệu raw video + metadata
   * tạo release skeleton
   * tạo media mapping
   * chia batch/work units cho worker notebooks

5. Đẩy phase00 release lên Hugging Face Dataset để worker notebooks dùng tiếp.

Notebook này **không xử lý structure/features/merge**. Các bước đó nằm ở Notebook 01/02/03.

Nguyên tắc:

* Người dùng chỉ chỉnh **Section 1 — User config**.
* Các cell phía dưới không nên sửa nếu không debug.
* Notebook chỉ orchestration; logic thật nằm trong package `system1`.
* Phase00 release được sync/restore thông qua CLI, notebook không tự zip/unzip checkpoint thủ công.


# Section 1 — User config

Đây là nơi duy nhất người dùng thường chỉnh trước khi chạy notebook. Mỗi nhóm config bên dưới tương ứng với một quyết định cụ thể: clone repo nào, đọc input từ đâu, chạy mode nào, và đẩy phase00 release lên HF Dataset nào.

Nếu bạn chạy trên Colab với dataset ban tổ chức đang nằm trong Google Drive, chỉ cần nhìn 4 chỗ: GitHub branch, workflow chuẩn ở Section 1.2, execution mode, và số batch. Các section sau chủ yếu là setup và chạy lệnh; không cần sửa nếu không debug.

Notebook 00 chỉ chạy phase00: ingest dữ liệu đầu vào, tạo batch assignment, rồi sync release lên HF Dataset. Không zip/unzip thủ công trong notebook.


## 1.1 GitHub repo config

Hai biến trong phần này là:

- `GITHUB_REPO_URL`: URL repo sẽ được clone hoặc pull.
- `GITHUB_BRANCH`: branch notebook sẽ checkout trước khi chạy.

Repo public thì giữ nguyên cấu hình mặc định. Nếu repo private, dùng `GITHUB_TOKEN` qua Kaggle Secret, Colab Secret, hoặc biến môi trường local; không hardcode token trong notebook.

`GITHUB_BRANCH` hiện nên là `system1`; sau khi nhánh được merge thì có thể đổi thành `main`.

Khi nào cần đổi:

- Đổi `GITHUB_REPO_URL` khi bạn fork repo hoặc dùng mirror riêng.
- Đổi `GITHUB_BRANCH` khi muốn test branch feature cụ thể.
- Nếu chỉ chạy workflow chuẩn hiện tại, cứ giữ mặc định.


In [ ]:
from __future__ import annotations

import os

GITHUB_REPO_URL = os.environ.get(
    "GITHUB_REPO_URL",
    "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git",
)
GITHUB_BRANCH = os.environ.get("GITHUB_BRANCH", "system1")


## 1.2 Workflow chuẩn: Drive zip -> standardized input -> phase00 -> HF Dataset

Notebook 00 dùng một workflow chính theo thứ tự cố định:

1. Copy folder Google Drive nguồn sang folder Drive của bạn bằng API cloud-to-cloud.
2. Extract/flatten toàn bộ zip từ folder đã copy thành layout chuẩn `raw_videos/` và `metadata/`.
3. Kiểm tra input chuẩn.
4. Chạy ingest trên layout chuẩn đó.
5. Chia batch cho worker notebooks.
6. Đẩy phase00 release lên Hugging Face Dataset để các notebook sau restore/load từ đó.

Bạn chỉ cần set các biến dưới đây khi dùng Colab/Drive:

- `AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID`: ID folder Drive nguồn của ban tổ chức.
- `AIC_DRIVE_SHADOW_TARGET_FOLDER_ID`: ID folder Drive đích thuộc Drive của bạn.
- `AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR`: đường dẫn mounted Drive tới folder zip sau khi shadow/copy.
- `AIC_STANDARDIZE_ARCHIVE_TARGET_DIR`: folder output chuẩn hóa; để trống thì dùng `AIC_DATA_ROOT`.
- `AIC_HF_REPO_ID`: Hugging Face Dataset repo dùng làm release store cho Notebook 01/02/03.

Notebook dùng mặc định production-safe: Drive shadow và archive standardization fail-fast nếu report có lỗi item-level; rerun sẽ skip output đã tồn tại với cùng size thay vì copy/flatten lại.

Fallback duy nhất: nếu bạn đã có sẵn input đúng layout, để trống các biến Drive/archive và set `AIC_DATA_ROOT`. Notebook vẫn ingest, assign batch, rồi đẩy release lên `AIC_HF_REPO_ID`.



In [ ]:
# Workflow chuẩn hiện tại: Drive zip -> standardized input -> phase00 -> HF Dataset.
AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID = os.environ.get("AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID", "")
AIC_DRIVE_SHADOW_TARGET_FOLDER_ID = os.environ.get("AIC_DRIVE_SHADOW_TARGET_FOLDER_ID", "")
AIC_DRIVE_SHADOW_REPORT_PATH = os.environ.get("AIC_DRIVE_SHADOW_REPORT_PATH", "")

AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR = os.environ.get("AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR", "")
AIC_STANDARDIZE_ARCHIVE_TARGET_DIR = os.environ.get("AIC_STANDARDIZE_ARCHIVE_TARGET_DIR", "")
AIC_STANDARDIZE_ARCHIVE_TEMP_DIR = os.environ.get("AIC_STANDARDIZE_ARCHIVE_TEMP_DIR", "")
AIC_STANDARDIZE_ARCHIVE_MEDIA_EXTENSIONS = os.environ.get("AIC_STANDARDIZE_ARCHIVE_MEDIA_EXTENSIONS", ".mp4,.mov,.mkv,.avi,.webm,.wav")
AIC_STANDARDIZE_ARCHIVE_OVERWRITE = os.environ.get("AIC_STANDARDIZE_ARCHIVE_OVERWRITE", "false")


## 1.3 Execution mode và batching

Hai biến trong phần này là:

- `AIC_EXECUTION_MODE`: mức chạy pipeline.
- `AIC_NUM_BATCHES`: số batch manifest tạm thời muốn tạo.

`AIC_EXECUTION_MODE` hiện có các lựa chọn hợp lệ:

- `debug_small_sample`: chạy nhanh nhất, phù hợp để smoke test notebook/end-to-end flow.
- `bronze_fast`: nhanh hơn chạy thật, dùng để kiểm tra pipeline trên tập lớn hơn debug.
- `silver_balanced`: cân bằng giữa tốc độ và độ đầy đủ.
- `gold_full`: gần nhất với chế độ chạy đầy đủ.

`AIC_NUM_BATCHES` là số work units/batch files muốn tạo tạm thời. Đây không phải số teammate và cũng không phải số account.

Gợi ý cấu hình:

- Debug notebook: `debug_small_sample`, `1`.
- Chạy thử nghiêm túc hơn: tăng execution mode trước, rồi mới tăng số batch.
- Chạy thật nhiều worker: có thể tạo nhiều batch hơn số worker dự kiến để dễ retry và phân phối lại.

Về sau phần này có thể nâng cấp thành cost-based batching, nhưng ở notebook này nó vẫn là cấu hình thủ công dễ hiểu.


In [ ]:
AIC_EXECUTION_MODE = os.environ.get("AIC_EXECUTION_MODE", "debug_small_sample")
AIC_NUM_BATCHES = int(os.environ.get("AIC_NUM_BATCHES", "1"))

## 1.4 Release identity

Biến trong phần này là:

- `AIC_RELEASE_ID`: tên release folder dưới output.

`AIC_RELEASE_ID` nên đổi khi bạn muốn tách nhiều lần chạy ra các release khác nhau, ví dụ:

- `competition_dataset_v001`
- `competition_dataset_debug_v002`
- `competition_dataset_teamA_trial`


In [ ]:
AIC_RELEASE_ID = os.environ.get("AIC_RELEASE_ID", "competition_dataset_v001")

## 1.5 Path overrides

Nhóm này cho phép override path mặc định mà notebook tự suy ra theo môi trường.

Các biến bạn có thể set là:

- `AIC_WORKSPACE_ROOT_OVERRIDE`
- `AIC_REPO_PARENT_OVERRIDE`
- `AIC_REPO_ROOT_OVERRIDE`
- `AIC_INPUT_ROOT_OVERRIDE`
- `AIC_OUTPUT_ROOT_OVERRIDE`
- `AIC_ARTIFACT_ROOT_OVERRIDE`

Để trống thì notebook tự chọn path theo môi trường.

- Kaggle mặc định dùng `/kaggle/working/...`
- Colab mặc định dùng `/content/...`
- Local dùng workspace/repo local hiện tại

Khi nào nên override:

- Bạn mount dữ liệu ở path khác mặc định.
- Bạn muốn output nằm ở disk khác.
- Bạn muốn repo clone vào thư mục riêng thay vì workspace mặc định.

Phần này giữ cả tên env mới và legacy env để tương thích, nên nếu project cũ còn dùng `AIC_DATA_ROOT` hoặc `AIC_RUNTIME_ROOT` thì vẫn map được.


In [ ]:
AIC_WORKSPACE_ROOT_OVERRIDE = os.environ.get("AIC_WORKSPACE_ROOT", "")
AIC_REPO_PARENT_OVERRIDE = os.environ.get("AIC_REPO_PARENT", "")
AIC_REPO_ROOT_OVERRIDE = os.environ.get("AIC_REPO_ROOT", "")

AIC_INPUT_ROOT_OVERRIDE = os.environ.get(
    "AIC_INPUT_ROOT",
    os.environ.get("AIC_DATA_ROOT", ""),
)
AIC_OUTPUT_ROOT_OVERRIDE = os.environ.get(
    "AIC_OUTPUT_ROOT",
    os.environ.get("AIC_RUNTIME_ROOT", ""),
)
AIC_ARTIFACT_ROOT_OVERRIDE = os.environ.get("AIC_ARTIFACT_ROOT", "")


## 1.6 Hugging Face release dataset

Nhóm này quyết định phase00 release sẽ được đẩy lên Hugging Face Dataset repo nào sau bước assign batch.

Các biến chính là:

- `AIC_HF_REPO_ID`: repo Hugging Face Dataset bắt buộc để Notebook 01/02/03 restore/load phase00 output.
- `AIC_HF_REPO_TYPE`: mặc định là `dataset`.
- `AIC_HF_REVISION`: branch/revision trên Hugging Face.
- `AIC_HF_PREFIX`: prefix logical để tách release hoặc run; mặc định bằng `AIC_RELEASE_ID`.

Repo HF phải được tạo sẵn và token phải có quyền write. Token không được hardcode; dùng `AIC_HF_TOKEN` hoặc `HF_TOKEN` qua Colab secret/env.

Sau Notebook 00, các notebook worker dùng cùng `AIC_HF_REPO_ID`, `AIC_HF_PREFIX`, và `AIC_RELEASE_ID` để restore phase00 output từ HF Dataset.


In [ ]:
AIC_HF_REPO_ID = os.environ.get("AIC_HF_REPO_ID", "")
AIC_HF_REPO_TYPE = os.environ.get("AIC_HF_REPO_TYPE", "dataset")
AIC_HF_REVISION = os.environ.get("AIC_HF_REVISION", "main")
AIC_HF_PREFIX = os.environ.get("AIC_HF_PREFIX", AIC_RELEASE_ID)


## 1.7 Resume/sync behavior

Hai biến trong phần này là:

- `AIC_RESUME`: có thử restore checkpoint trước khi chạy phase hay không.
- `AIC_SYNC`: có save checkpoint sau khi phase chạy xong hay không.

Ý nghĩa thực tế:

- `AIC_RESUME=true`: restore checkpoint nếu có, giúp bỏ qua phần đã làm xong.
- `AIC_SYNC=true`: save checkpoint sau khi phase00 hoàn tất để runtime khác có thể dùng lại.

Gợi ý dùng:

- Debug local ngắn: có thể để `resume=false`, `sync=false`.
- Chạy nhiều runtime chung backend: thường để `resume=true`, `sync=true`.

Section này chỉ cấu hình; logic restore/save nằm ở các section chạy CLI sau.


In [ ]:
AIC_RESUME = os.environ.get("AIC_RESUME", "true")
AIC_SYNC = os.environ.get("AIC_SYNC", "true")

## 1.8 Derived config, validation, export env

Cell này không dành cho người dùng chỉnh.

Nó làm 3 việc:

1. Chuẩn hóa alias cũ/mới để các section sau không bị vỡ.
2. Validate config sớm để lỗi lộ ra ngay ở đầu notebook.
3. Export env để CLI subprocess đọc cùng một bộ config.

Nếu notebook fail ngay ở cell này, thường là do:

- execution mode không hợp lệ,
- chưa set `AIC_HF_REPO_ID` cho HF release dataset,
- bật Drive shadow nhưng chưa chỉ rõ folder zip để flatten,
- hoặc `AIC_NUM_BATCHES < 1`.


In [ ]:
execution_mode = AIC_EXECUTION_MODE
num_batches = AIC_NUM_BATCHES

SUPPORTED_EXECUTION_MODES = {
    "debug_small_sample",
    "bronze_fast",
    "silver_balanced",
    "gold_full",
}

def parse_bool(value: str | bool) -> bool:
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "yes", "y", "on"}:
        return True
    if normalized in {"0", "false", "no", "n", "off"}:
        return False
    raise ValueError(f"Giá trị boolean không hợp lệ: {value!r}")


AIC_RESUME_ENABLED = parse_bool(AIC_RESUME)
AIC_SYNC_ENABLED = parse_bool(AIC_SYNC)
AIC_STANDARDIZE_ARCHIVE_OVERWRITE_ENABLED = parse_bool(AIC_STANDARDIZE_ARCHIVE_OVERWRITE)

if execution_mode not in SUPPORTED_EXECUTION_MODES:
    raise ValueError(
        f"AIC_EXECUTION_MODE không hợp lệ: {execution_mode}. "
        f"Chọn một trong: {sorted(SUPPORTED_EXECUTION_MODES)}"
    )

if not AIC_HF_REPO_ID:
    raise ValueError("Workflow chuẩn cần set AIC_HF_REPO_ID để đẩy phase00 release lên Hugging Face Dataset.")

if os.environ.get("AIC_ORGANIZER_SOURCE_URI") or os.environ.get("AIC_CANONICAL_HF_REPO_ID"):
    raise ValueError(
        "Notebook 00 chuẩn không dùng AIC_ORGANIZER_SOURCE_URI/AIC_CANONICAL_HF_REPO_ID. "
        "Hãy dùng Drive shadow -> standardize archives -> local ingest -> sync-release lên AIC_HF_REPO_ID."
    )

if bool(AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID) != bool(AIC_DRIVE_SHADOW_TARGET_FOLDER_ID):
    raise ValueError("Drive shadow cần set cả AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID và AIC_DRIVE_SHADOW_TARGET_FOLDER_ID.")

if AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID and not AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR:
    raise ValueError(
        "Workflow chuẩn Drive cần set AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR tới folder zip đã mount, "
        "ví dụ /content/drive/MyDrive/AIC2026/raw_dataset."
    )

if num_batches < 1:
    raise ValueError("AIC_NUM_BATCHES phải >= 1.")

if AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID or AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR:
    AIC_INPUT_PLAN = "drive_zip_to_standardized_input"
else:
    AIC_INPUT_PLAN = "prestandardized_local_input"


def export_env(name: str, value: object) -> None:
    os.environ[name] = str(value)


def export_env_if_set(name: str, value: object) -> None:
    value_text = str(value).strip()
    if value_text:
        os.environ[name] = value_text


export_env("GITHUB_REPO_URL", GITHUB_REPO_URL)
export_env("GITHUB_BRANCH", GITHUB_BRANCH)
export_env_if_set("AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID", AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID)
export_env_if_set("AIC_DRIVE_SHADOW_TARGET_FOLDER_ID", AIC_DRIVE_SHADOW_TARGET_FOLDER_ID)
export_env_if_set("AIC_DRIVE_SHADOW_REPORT_PATH", AIC_DRIVE_SHADOW_REPORT_PATH)
export_env_if_set("AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR", AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR)
export_env_if_set("AIC_STANDARDIZE_ARCHIVE_TARGET_DIR", AIC_STANDARDIZE_ARCHIVE_TARGET_DIR)
export_env_if_set("AIC_STANDARDIZE_ARCHIVE_TEMP_DIR", AIC_STANDARDIZE_ARCHIVE_TEMP_DIR)
export_env("AIC_STANDARDIZE_ARCHIVE_MEDIA_EXTENSIONS", AIC_STANDARDIZE_ARCHIVE_MEDIA_EXTENSIONS)
export_env("AIC_STANDARDIZE_ARCHIVE_OVERWRITE", "true" if AIC_STANDARDIZE_ARCHIVE_OVERWRITE_ENABLED else "false")

export_env("AIC_EXECUTION_MODE", execution_mode)
export_env("AIC_NUM_BATCHES", num_batches)

export_env("AIC_RELEASE_ID", AIC_RELEASE_ID)

export_env_if_set("AIC_WORKSPACE_ROOT", AIC_WORKSPACE_ROOT_OVERRIDE)
export_env_if_set("AIC_REPO_PARENT", AIC_REPO_PARENT_OVERRIDE)
export_env_if_set("AIC_REPO_ROOT", AIC_REPO_ROOT_OVERRIDE)

export_env_if_set("AIC_INPUT_ROOT", AIC_INPUT_ROOT_OVERRIDE)
export_env_if_set("AIC_OUTPUT_ROOT", AIC_OUTPUT_ROOT_OVERRIDE)
export_env_if_set("AIC_DATA_ROOT", AIC_INPUT_ROOT_OVERRIDE)
export_env_if_set("AIC_RUNTIME_ROOT", AIC_OUTPUT_ROOT_OVERRIDE)
export_env_if_set("AIC_ARTIFACT_ROOT", AIC_ARTIFACT_ROOT_OVERRIDE)

export_env("AIC_HF_REPO_ID", AIC_HF_REPO_ID)
export_env("AIC_HF_REPO_TYPE", AIC_HF_REPO_TYPE)
export_env("AIC_HF_REVISION", AIC_HF_REVISION)
export_env("AIC_HF_PREFIX", AIC_HF_PREFIX)

export_env("AIC_RESUME", "true" if AIC_RESUME_ENABLED else "false")
export_env("AIC_SYNC", "true" if AIC_SYNC_ENABLED else "false")

print({
    "input_plan": AIC_INPUT_PLAN,
    "github_branch": GITHUB_BRANCH,
    "execution_mode": execution_mode,
    "num_batches": num_batches,
    "release_id": AIC_RELEASE_ID,
    "drive_shadow_enabled": bool(AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID),
    "standardize_archives_enabled": bool(AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR),
    "hf_repo_id": AIC_HF_REPO_ID,
    "hf_prefix": AIC_HF_PREFIX,
    "resume": AIC_RESUME_ENABLED,
    "sync": AIC_SYNC_ENABLED,
})


# Section 2 — Detect môi trường và thiết lập workspace

Cell này tự nhận diện môi trường chạy.

- Kaggle dùng `/kaggle/working` làm workspace mặc định.
- Colab dùng `/content` làm workspace mặc định; nếu muốn output bền vững thì mount Google Drive.
- Local dùng thư mục hiện tại hoặc `AIC_WORKSPACE_ROOT`.

In [ ]:
from pathlib import Path


def detect_environment() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab  # type: ignore  # noqa: F401
        return "colab"
    except Exception:
        return "local"


ENVIRONMENT = detect_environment()
if ENVIRONMENT == "kaggle":
    default_workspace_root = Path("/kaggle/working")
elif ENVIRONMENT == "colab":
    default_workspace_root = Path("/content")
else:
    default_workspace_root = Path.cwd()

WORKSPACE_ROOT = Path(os.environ.get("AIC_WORKSPACE_ROOT", str(default_workspace_root))).expanduser().resolve()
AIC_REPO_PARENT = Path(os.environ.get("AIC_REPO_PARENT", str(WORKSPACE_ROOT))).expanduser().resolve()
AIC_REPO_PARENT.mkdir(parents=True, exist_ok=True)

print({
    "environment": ENVIRONMENT,
    "workspace_root": str(WORKSPACE_ROOT),
    "AIC_REPO_PARENT": str(AIC_REPO_PARENT),
})

# Section 3 — Secret và repo private

Nếu repo public thì không cần set secret.

Nếu repo private:

- Kaggle: Add-ons / Secrets → tạo `GITHUB_TOKEN`.
- Colab: Secrets → tạo `GITHUB_TOKEN`.
- Local: `export GITHUB_TOKEN=...`.

Không hardcode token vào notebook và không print token ra log.

Nếu bật Drive shadow hoặc dùng path dưới `/content/drive`, cell này cũng mount Google Drive. Nếu bật Drive shadow trên Colab, cell này chạy `auth.authenticate_user()` để CLI `system1 drive-shadow` dùng được Google Drive API credentials.

In [ ]:
from urllib.parse import urlparse, urlunparse


def get_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    if ENVIRONMENT == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    if ENVIRONMENT == "colab":
        try:
            from google.colab import userdata  # type: ignore
            return userdata.get(name)
        except Exception:
            return None
    return None


def auth_repo_url(url: str) -> str:
    token = get_secret("GITHUB_TOKEN")
    parsed = urlparse(url)
    if token and parsed.scheme == "https" and parsed.netloc == "github.com":
        return urlunparse((parsed.scheme, f"x-access-token:{token}@{parsed.netloc}", parsed.path, "", "", ""))
    return url


def safe_repo_url(url: str) -> str:
    parsed = urlparse(url)
    if "@" in parsed.netloc:
        safe_netloc = "***@" + parsed.netloc.split("@", 1)[1]
        return urlunparse((parsed.scheme, safe_netloc, parsed.path, "", "", ""))
    return url


def should_mount_colab_drive() -> bool:
    if ENVIRONMENT != "colab":
        return False
    drive_paths = [
        AIC_INPUT_ROOT_OVERRIDE,
        AIC_OUTPUT_ROOT_OVERRIDE,
        AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR,
        AIC_STANDARDIZE_ARCHIVE_TARGET_DIR,
    ]
    return bool(AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID) or any(str(path).startswith("/content/drive") for path in drive_paths if path)


if should_mount_colab_drive():
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

if ENVIRONMENT == "colab" and AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID:
    from google.colab import auth  # type: ignore
    auth.authenticate_user()

print("repo_url=", safe_repo_url(auth_repo_url(GITHUB_REPO_URL)))

# Section 4 — Clone/pull repo và checkout branch

Cell này clone repo nếu chưa có, hoặc fetch/checkout/pull đúng branch nếu repo đã tồn tại.

Lưu ý:

- Nếu đang sửa code trực tiếp trong runtime, `git pull --ff-only` có thể fail nếu có thay đổi local.
- Khi test branch hiện tại, giữ `GITHUB_BRANCH = "system1"`.
- Kaggle ưu tiên workspace `/kaggle/working`, không dùng `/content`.

In [ ]:
import shutil
import subprocess
import sys

REPO_DIR_NAME = Path(urlparse(GITHUB_REPO_URL).path).stem or "Multimodal-Agentic-Retrieval-Engine"


def run_shell(command: list[str], *, cwd: Path | None = None, safe_command: list[str] | None = None) -> None:
    shown = safe_command or command
    print("$", " ".join(str(part) for part in shown))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


def find_system1_root(repo_root: Path) -> Path | None:
    if (repo_root / "system1" / "pyproject.toml").exists():
        return repo_root / "system1"
    if (repo_root / "pyproject.toml").exists() and (repo_root / "src" / "system1").exists():
        return repo_root
    return None


def current_tree_repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if find_system1_root(candidate) is not None:
            return candidate
    return None


def resolve_repo_root() -> Path:
    env_repo_root = os.environ.get("AIC_REPO_ROOT")
    if env_repo_root:
        candidate = Path(env_repo_root).expanduser().resolve()
        if find_system1_root(candidate) is not None:
            return candidate
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "system1").exists():
            return candidate
        raise FileNotFoundError(f"AIC_REPO_ROOT không chứa package system1 hợp lệ: {candidate}")

    local_repo_root = current_tree_repo_root()
    if ENVIRONMENT == "local" and local_repo_root is not None:
        return local_repo_root

    return AIC_REPO_PARENT / REPO_DIR_NAME


REPO_ROOT = resolve_repo_root()
SYSTEM1_ROOT = find_system1_root(REPO_ROOT)
clone_url = auth_repo_url(GITHUB_REPO_URL)
safe_clone_url = safe_repo_url(clone_url)

if SYSTEM1_ROOT is not None and not (REPO_ROOT / ".git").exists():
    print(f"Dùng repo/package local có sẵn: {REPO_ROOT}")
elif REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    # Nếu repo đã tồn tại và là git repo, cập nhật đúng branch thay vì clone lại.
    run_shell(["git", "fetch", "origin"], cwd=REPO_ROOT)
    run_shell(["git", "checkout", GITHUB_BRANCH], cwd=REPO_ROOT)
    run_shell(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=REPO_ROOT)
elif REPO_ROOT.exists() and find_system1_root(REPO_ROOT) is None:
    # Nếu folder clone bị lỗi giữa chừng, xóa để tránh dùng trạng thái hỏng.
    shutil.rmtree(REPO_ROOT)
    run_shell(
        ["git", "clone", "--branch", GITHUB_BRANCH, clone_url, str(REPO_ROOT)],
        safe_command=["git", "clone", "--branch", GITHUB_BRANCH, safe_clone_url, str(REPO_ROOT)],
    )
elif not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run_shell(
        ["git", "clone", "--branch", GITHUB_BRANCH, clone_url, str(REPO_ROOT)],
        safe_command=["git", "clone", "--branch", GITHUB_BRANCH, safe_clone_url, str(REPO_ROOT)],
    )

SYSTEM1_ROOT = find_system1_root(REPO_ROOT)
if SYSTEM1_ROOT is None:
    raise FileNotFoundError(f"Không tìm thấy package system1 trong repo: {REPO_ROOT}")

print({"repo_root": str(REPO_ROOT), "system1_root": str(SYSTEM1_ROOT), "github_branch": GITHUB_BRANCH})

# Section 5 — Cài package System 1

Cell này cài package bằng editable install.

- `-e` nghĩa là editable install: notebook gọi CLI `system1`, nhưng code chạy từ package vừa cài.
- `pydrive2` cung cấp Google Drive client dependencies cho workflow Drive shadow.
- `SYSTEM1_ROOT` phải là package root có `pyproject.toml`.

In [ ]:
if not (SYSTEM1_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"SYSTEM1_ROOT không có pyproject.toml: {SYSTEM1_ROOT}")

run_shell([sys.executable, "-m", "pip", "install", "-q", "gdown", "pydrive2", "-e", str(SYSTEM1_ROOT)])

os.environ["AIC_REPO_ROOT"] = str(REPO_ROOT)
os.environ["AIC_SYSTEM1_ROOT"] = str(SYSTEM1_ROOT)

# Section 6 — Thiết lập data/output/runtime paths

Cell này tạo các biến đường dẫn runtime.

- Kaggle mặc định dùng `AIC_DATA_ROOT = /kaggle/working/input`, `AIC_RUNTIME_ROOT = /kaggle/working/output`.
- Colab mặc định dùng `AIC_DATA_ROOT = /content/input`, `AIC_RUNTIME_ROOT = /content/output`.
- Local mặc định dùng `SYSTEM1_ROOT / "input"` và `SYSTEM1_ROOT / "output"`.

Ví dụ:

Kaggle Dataset attach:

```python
os.environ["AIC_DATA_ROOT"] = "/kaggle/input/<dataset-name>"
os.environ["AIC_RUNTIME_ROOT"] = "/kaggle/working/output"
```

Colab Drive:

```python
from google.colab import drive
drive.mount("/content/drive")
os.environ["AIC_DATA_ROOT"] = "/content/drive/MyDrive/aic/system1/input"
os.environ["AIC_RUNTIME_ROOT"] = "/content/drive/MyDrive/aic/system1/output"
```

Local:

```python
os.environ["AIC_REPO_ROOT"] = "/path/to/Multimodal-Agentic-Retrieval-Engine"
os.environ["AIC_DATA_ROOT"] = "/path/to/input"
os.environ["AIC_RUNTIME_ROOT"] = "/path/to/output"
```

In [ ]:
if ENVIRONMENT == "kaggle":
    default_data_root = Path("/kaggle/working/input")
    default_runtime_root = Path("/kaggle/working/output")
elif ENVIRONMENT == "colab":
    default_data_root = Path("/content/input")
    default_runtime_root = Path("/content/output")
else:
    default_data_root = SYSTEM1_ROOT / "input"
    default_runtime_root = SYSTEM1_ROOT / "output"

AIC_DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", str(default_data_root))).expanduser().resolve()
AIC_RUNTIME_ROOT = Path(os.environ.get("AIC_RUNTIME_ROOT", str(default_runtime_root))).expanduser().resolve()
AIC_ARTIFACT_ROOT = Path(os.environ.get("AIC_ARTIFACT_ROOT", str(AIC_RUNTIME_ROOT))).expanduser().resolve()
STANDARDIZE_ARCHIVE_SOURCE_DIR = Path(AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR).expanduser().resolve() if AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR else None
STANDARDIZE_ARCHIVE_TARGET_DIR = Path(AIC_STANDARDIZE_ARCHIVE_TARGET_DIR).expanduser().resolve() if AIC_STANDARDIZE_ARCHIVE_TARGET_DIR else AIC_DATA_ROOT
STANDARDIZE_ARCHIVE_TEMP_DIR = Path(AIC_STANDARDIZE_ARCHIVE_TEMP_DIR).expanduser().resolve() if AIC_STANDARDIZE_ARCHIVE_TEMP_DIR else (WORKSPACE_ROOT / "archive_extraction")
DRIVE_SHADOW_REPORT_PATH = Path(AIC_DRIVE_SHADOW_REPORT_PATH).expanduser().resolve() if AIC_DRIVE_SHADOW_REPORT_PATH else (AIC_RUNTIME_ROOT / "drive_shadow_report.json")

input_dir = AIC_DATA_ROOT
output_dir = AIC_RUNTIME_ROOT
release_dir = output_dir / AIC_RELEASE_ID
RELEASE_DIR = release_dir

AIC_DATA_ROOT.mkdir(parents=True, exist_ok=True)
AIC_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
AIC_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
STANDARDIZE_ARCHIVE_TARGET_DIR.mkdir(parents=True, exist_ok=True)
STANDARDIZE_ARCHIVE_TEMP_DIR.mkdir(parents=True, exist_ok=True)

os.environ["AIC_DATA_ROOT"] = str(AIC_DATA_ROOT)
os.environ["AIC_RUNTIME_ROOT"] = str(AIC_RUNTIME_ROOT)
os.environ["AIC_ARTIFACT_ROOT"] = str(AIC_ARTIFACT_ROOT)

print({
    "input_dir": str(input_dir),
    "output_dir": str(output_dir),
    "release_dir": str(release_dir),
    "standardize_archive_source_dir": str(STANDARDIZE_ARCHIVE_SOURCE_DIR) if STANDARDIZE_ARCHIVE_SOURCE_DIR else None,
    "standardize_archive_target_dir": str(STANDARDIZE_ARCHIVE_TARGET_DIR),
})

# Section 7 — Helper: chạy CLI và đọc JSON

Cell này định nghĩa helper dùng chung.

- `run_cli` luôn in command trước khi chạy.
- Command chạy ở `cwd=SYSTEM1_ROOT`.
- `check=True` giúp fail fast nếu CLI lỗi.

In [ ]:
import json
from typing import Any


def system1_executable() -> list[str]:
    binary = shutil.which("system1")
    if binary:
        return [binary]
    return [sys.executable, "-m", "system1.cli"]


def run_cli(args: list[str]) -> None:
    command = [*system1_executable(), *args]
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=str(SYSTEM1_ROOT), check=True)


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def path_summary() -> None:
    print({
        "environment": ENVIRONMENT,
        "repo_root": str(REPO_ROOT),
        "system1_root": str(SYSTEM1_ROOT),
        "github_branch": GITHUB_BRANCH,
        "input_dir": str(input_dir),
        "output_dir": str(output_dir),
        "release_dir": str(release_dir),
        "execution_mode": execution_mode,
        "num_batches": num_batches,
    })


path_summary()

# Section 8 — Chuẩn bị input chuẩn

Cell này chạy đúng một input plan đã validate ở Section 1.8.

Plan chuẩn cho Colab/Drive là `drive_zip_to_standardized_input`:

1. `drive-shadow`: copy folder Drive nguồn sang Drive đích.
2. `standardize-archives`: extract zip và flatten media/JSON vào `raw_videos/` + `metadata/`.
3. Kiểm tra `AIC_DATA_ROOT` đã có layout chuẩn trước khi ingest.

Hai lệnh input-prep fail-fast khi report có lỗi. Không chạy ingest trên input chuẩn bị dang dở trừ khi bạn chủ động chạy CLI ngoài notebook với `--allow-partial` để recovery thủ công.

Layout chuẩn sau bước chuẩn bị input là:

```text
raw_videos/
metadata/
```


In [ ]:
def input_ready(data_root: Path) -> bool:
    return (data_root / "raw_videos").exists() and (data_root / "metadata").exists()


print(f"Input plan: {AIC_INPUT_PLAN}")

if AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID:
    print("Step 1/3: Drive shadow copy")
    run_cli([
        "drive-shadow",
        "--source-folder-id", AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID,
        "--dest-folder-id", AIC_DRIVE_SHADOW_TARGET_FOLDER_ID,
        "--report-path", str(DRIVE_SHADOW_REPORT_PATH),
    ])

if STANDARDIZE_ARCHIVE_SOURCE_DIR:
    print("Step 2/3: Standardize archive source")
    run_cli([
        "standardize-archives",
        "--source-dir", str(STANDARDIZE_ARCHIVE_SOURCE_DIR),
        "--target-dir", str(STANDARDIZE_ARCHIVE_TARGET_DIR),
        "--temp-dir", str(STANDARDIZE_ARCHIVE_TEMP_DIR),
        "--media-extensions", AIC_STANDARDIZE_ARCHIVE_MEDIA_EXTENSIONS,
        "--overwrite" if AIC_STANDARDIZE_ARCHIVE_OVERWRITE_ENABLED else "--no-overwrite",
    ])

if input_ready(AIC_DATA_ROOT):
    print(f"Step 3/3: Input chuẩn đã sẵn sàng tại {AIC_DATA_ROOT}")
else:
    raise FileNotFoundError(
        "Chưa có input dataset chuẩn sau bước chuẩn bị. Hãy kiểm tra lại config workflow chuẩn:\n"
        f"{AIC_DATA_ROOT}/\n"
        "  raw_videos/\n"
        "  metadata/\n"
        "Với Colab/Drive, cần set Drive source/target IDs và AIC_STANDARDIZE_ARCHIVE_SOURCE_DIR tới folder zip đã mount."
    )


# Section 9 — Chạy ingest

`ingest` đọc raw video + metadata và tạo các output chính:

- `tables/videos.parquet`
- `raw_mapping/media_store_manifest.parquet`
- `manifests/dataset_report.json`

Nếu fail, kiểm tra lại input layout và metadata pairing.

In [ ]:
run_cli([
    "ingest",
    "--mode", execution_mode,
    "--input", str(input_dir),
    "--output", str(output_dir),
])


# Section 10 — Chia batch

`assign-batches` tạo:

- `manifests/batch_manifest.csv`
- `manifests/batch_000.txt`
- các file batch tiếp theo nếu `AIC_NUM_BATCHES > 1`

`AIC_NUM_BATCHES` nên bằng số work units cần tạo. Notebook 01 và 02 chọn `batch_id` từ các file `batch_*.txt` được sinh ra.

In [ ]:
run_cli([
    "assign-batches",
    "--mode", execution_mode,
    "--num-batches", str(num_batches),
    "--output", str(output_dir),
])

# Section 11 — Đẩy phase00 release lên Hugging Face Dataset

Cell này upload output phase00 lên `AIC_HF_REPO_ID` dưới `releases/<AIC_RELEASE_ID>/...`. Đây là bước bắt buộc của workflow chuẩn: Notebook 01/02/03 sẽ restore/load phase00 output từ HF Dataset này, không phụ thuộc vào runtime local của Notebook 00.


In [ ]:
run_cli([
    "sync-release",
    "--output", str(output_dir),
    "--hf-repo-id", AIC_HF_REPO_ID,
    "--hf-prefix", AIC_HF_PREFIX,
    "--hf-repo-type", AIC_HF_REPO_TYPE,
    "--hf-revision", AIC_HF_REVISION,
])


# Section 12 — Kiểm tra nhanh kết quả

User cần kiểm tra:

- số video có đúng không;
- `video_id` có đúng filename stem không;
- batch files có được tạo không;
- nếu dùng nhiều worker, phân batch có đúng số lượng không.

Report section xử lý thiếu file bằng cảnh báo tiếng Việt. Riêng `videos.parquet` thiếu thì báo lỗi rõ vì ingest chưa chạy hoặc đã fail.

In [ ]:
import csv

import pandas as pd
from IPython.display import JSON, display


def show_json_if_exists(path: Path, title: str) -> None:
    if path.exists():
        print(title)
        display(JSON(load_json(path)))
    else:
        print(f"Cảnh báo: chưa thấy {path}")


if AIC_DRIVE_SHADOW_SOURCE_FOLDER_ID:
    show_json_if_exists(DRIVE_SHADOW_REPORT_PATH, "drive_shadow_report.json")

if STANDARDIZE_ARCHIVE_SOURCE_DIR:
    show_json_if_exists(STANDARDIZE_ARCHIVE_TARGET_DIR / "standardize_archives_report.json", "standardize_archives_report.json")

organizer_report_path = input_dir / "organizer_import_report.json"
show_json_if_exists(organizer_report_path, "organizer_import_report.json")
print("phase00 release đã sync lên HF Dataset:", AIC_HF_REPO_ID)
print("hf_prefix=", AIC_HF_PREFIX)

videos_path = release_dir / "tables" / "videos.parquet"
if not videos_path.exists():
    raise FileNotFoundError(f"Không thấy {videos_path}. Có thể ingest chưa chạy hoặc đã fail.")

videos = pd.read_parquet(videos_path)
print(f"videos_count={len(videos)}")
preferred_columns = [column for column in ["video_id", "video_ref"] if column in videos.columns]
if preferred_columns:
    display(videos[preferred_columns])
else:
    display(videos.head())

show_json_if_exists(release_dir / "manifests" / "dataset_report.json", "dataset_report.json")
show_json_if_exists(release_dir / "manifests" / "dataset_manifest.json", "dataset_manifest.json")

batch_manifest_path = release_dir / "manifests" / "batch_manifest.csv"
if batch_manifest_path.exists():
    with batch_manifest_path.open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    print(f"batch_manifest_rows={len(rows)}")
    display(pd.DataFrame(rows))
else:
    print(f"Cảnh báo: chưa thấy {batch_manifest_path}")

batch_paths = sorted((release_dir / "manifests").glob("batch_*.txt"))
if not batch_paths:
    print("Cảnh báo: chưa thấy file batch_*.txt")
for path in batch_paths:
    print(f"--- {path.name} ---")
    print(path.read_text(encoding="utf-8").strip())

# Section 13 — Gợi ý bước tiếp theo

Sau khi Notebook 00 pass, chạy tiếp:

```text
01_worker_structure_pipeline.ipynb
02_worker_feature_enrichment.ipynb
03_merge_validate_index_release.ipynb
```

Các notebook sau dùng phase00 output từ Hugging Face Dataset đã sync ở Section 11:

- cùng `AIC_HF_REPO_ID`;
- cùng `AIC_HF_PREFIX`;
- cùng `AIC_RELEASE_ID`.

Với debug test một worker, worker notebook restore release rồi dùng batch đầu tiên, thường là `batch_000`.

Nếu chạy nhiều worker, mỗi worker restore cùng release từ HF Dataset rồi dùng một file `batch_*.txt` khác nhau.